# Extrinsic Downstream Task Evaluation — zephyr-7b-beta
## Sinhala Script Robustness — Unicode vs Romanized

**Model:** `HuggingFaceH4/zephyr-7b-beta`  
**Prompt template:** `T1_direct`  
**Reasoning mode:** `none`  
**8-bit quantization:** `False`

## 1. Configuration

In [1]:
# ============================================================
# MODEL CONFIGURATION — pre-filled for this model
# ============================================================

MODEL_ID    = "HuggingFaceH4/zephyr-7b-beta"    # HuggingFace model ID
MODEL_LABEL = "zephyr-7b-beta"              # Short label for filenames & tables
PROMPT_TEMPLATE = "T1_direct"  # Best template from prompt-evaluation pilot

# ---- Reasoning-mode flag ----
# Options: "none" | "no_think" | "enable_thinking_false"
# Applies identically to both script conditions and all datasets.
# No chain-of-thought suppression needed for this model.
REASONING_MODE = "none"

MAX_NEW_TOKENS = 40                 # Max tokens to generate per response
TEMPERATURE    = 0.0                # Greedy decoding (deterministic, zero-shot)
MAX_INPUT_TOKENS = 4096             # Truncate prompts longer than this (safety margin)

# Set True only for large models that won't fit in fp16 across 2x T4 (32 GB total).
LOAD_IN_8BIT = False

# If a previous run's CSV for this model+dataset already exists, skip ids already completed
# instead of starting over. Protects against Kaggle session interruptions on a ~9,479-item run.
RESUME = True

# Kaggle maximum runtime safety limit in hours (Kaggle hard timeout is 12.0h).
# If reached, the notebook gracefully saves all CSV progress, computes summary statistics,
# and finishes cleanly as SUCCESS rather than failing from timeout.
TIME_LIMIT_HOURS = 11.5

# Dataset paths (update if your Kaggle dataset mount differs)
DATA_DIR = "/kaggle/input/datasets/uom220131a/dataset2"  # Folder containing the 3 JSONL files

# Output prefix (auto-generated from MODEL_LABEL)
import re
OUTPUT_PREFIX = re.sub(r"[^a-z0-9]+", "_", MODEL_LABEL.lower()).strip("_")

print(f"Model:          {MODEL_ID}")
print(f"Label:          {MODEL_LABEL}")
print(f"Template:       {PROMPT_TEMPLATE}")
print(f"Reasoning mode: {REASONING_MODE}")
print(f"8-bit:          {LOAD_IN_8BIT}")
print(f"Resume:         {RESUME}")
print(f"Time limit:     {TIME_LIMIT_HOURS} hours")
print(f"Output:         {OUTPUT_PREFIX}_*.csv")

Model:          HuggingFaceH4/zephyr-7b-beta
Label:          zephyr-7b-beta
Template:       T1_direct
Reasoning mode: none
8-bit:          False
Resume:         True
Time limit:     11.5 hours
Output:         zephyr_7b_beta_*.csv


## 2. Environment Setup & Hardware Verification

In [2]:
import sys
import os

!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub bitsandbytes

import torch
import json
import csv
import time
import re
import transformers
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | {device_count} GPU(s)")
for i in range(device_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  cuda:{i} -> {name} ({mem:.1f} GB)")
assert device_count > 0, "No GPU detected. Enable GPU T4 x2 in Session options."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | transformers 5.15.1 | 2 GPU(s)
  cuda:0 -> Tesla T4 (14.6 GB)
  cuda:1 -> Tesla T4 (14.6 GB)


In [3]:
# HuggingFace authentication (needed for gated models, e.g. Llama-3.1-8B-Instruct)
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""   # Paste your token here if Kaggle Secrets is unavailable
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Authenticated via explicit token.")
    else:
        print(f"No HF_TOKEN found ({e}). Public models will still work.")

Authenticated via Kaggle Secrets.


## 3. Load Model & Tokenizer

In [4]:
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

has_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
print(f"Chat template available: {has_template}")
if not has_template:
    print("WARNING: this model has no chat template. It will be prompted with the raw text "
          "instead of its native assistant format — confirm this model is actually instruct-tuned "
          "before trusting the results (this notebook is not meant for base models).")
if not has_template and REASONING_MODE != "none":
    print(f"WARNING: REASONING_MODE={REASONING_MODE!r} but this model has no chat template. "
          "The reasoning-mode setting will have no effect.")

model_kwargs = dict(
    device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
if LOAD_IN_8BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
model.config.use_cache = True  # Enable KV cache for generation

print("Model loaded successfully.")
print(f"Precision: {model.dtype}")
if hasattr(model, "hf_device_map"):
    print(f"Device map: {model.hf_device_map}")

Loading HuggingFaceH4/zephyr-7b-beta...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded successfully.
Precision: torch.float16
Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.layers.28': 1, 'model.layers.29': 1, 'model.layers.30': 1, 'model.layers.31': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## 4. Load Evaluation Datasets

In [5]:
def load_jsonl(filepath):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

mmlu_data = load_jsonl(os.path.join(DATA_DIR, "sinhala_mmlu.jsonl"))
sold_data = load_jsonl(os.path.join(DATA_DIR, "sold.jsonl"))
piqa_data = load_jsonl(os.path.join(DATA_DIR, "global_piqa.jsonl"))

print(f"Sinhala MMLU : {len(mmlu_data):,} items (N-way MCQ)")
print(f"SOLD         : {len(sold_data):,} items (binary classification)")
print(f"Global PIQA  : {len(piqa_data):,} items (2-way MCQ)")
print(f"\nTotal items  : {len(mmlu_data) + len(sold_data) + len(piqa_data):,} "
      f"x 2 script conditions = {(len(mmlu_data) + len(sold_data) + len(piqa_data)) * 2:,} prompts")

Sinhala MMLU : 6,879 items (N-way MCQ)
SOLD         : 2,500 items (binary classification)
Global PIQA  : 100 items (2-way MCQ)

Total items  : 9,479 x 2 script conditions = 18,958 prompts


## 5. Prompt Templates & Label Extraction
Using template **`T1_direct`** (selected from prompt-evaluation pilot).
All three templates (T1/T2/T3) are defined below; only the one selected above is active.
Few-shot exemplars for T2 are embedded inline.

In [6]:
# ============================================================
# PROMPT TEMPLATES
# ============================================================
# Three interchangeable prompting techniques (T1/T2/T3) are defined below.
# PROMPT_TEMPLATE (set in the Configuration cell) selects which one is used
# for every dataset.

NO_REPEAT_INSTRUCTION = "Do not repeat the question or text in your response."


def _humanize_domain(domain):
    """'Social_Science' -> 'Social Science'. Leaves single-word domains untouched."""
    return str(domain).replace("_", " ")


def _mcq_options_block(options):
    digit_labels = [str(i + 1) for i in range(len(options))]
    options_str = "\n".join(f"{d}. {opt}" for d, opt in zip(digit_labels, options))
    return options_str, digit_labels


# ------------------------------------------------------------------
# T1 — DIRECT: answer only, nothing else
# ------------------------------------------------------------------

def build_mcq_prompt_t1(question, options, subject=None):
    options_str, digit_labels = _mcq_options_block(options)
    n = len(options)
    valid_range = "/".join(digit_labels)
    if subject is not None:
        prompt = (
            f"This is a multiple-choice question related to the {_humanize_domain(subject)}. "
            f"Choose the correct or most appropriate answer from answers {valid_range} for the "
            f"following question. {NO_REPEAT_INSTRUCTION} Respond with only the number "
            f"({valid_range}) and nothing else.\n"
            f"Question: {question}\n"
            f"{options_str}\n"
            f"Answer:"
        )
    else:
        prompt = (
            f"Out of the given {n} answers, which one is the best option to complete the given "
            f"text? {NO_REPEAT_INSTRUCTION} Respond with only the number ({valid_range}) and "
            f"nothing else.\n"
            f"{question}\n"
            f"{options_str}\n"
            f"Answer:"
        )
    return prompt, digit_labels


def build_sold_prompt_t1(text):
    return (
        f"Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' "
        f"(offensive). {NO_REPEAT_INSTRUCTION} Respond with only 'NOT' or 'OFF' and "
        f"nothing else.\n"
        f"Text: {text}\n"
        f"Answer:"
    )


# ------------------------------------------------------------------
# T2 — FEW-SHOT: 2 fixed exemplars answered in the exact target format
# ------------------------------------------------------------------

FEWSHOT_EXAMPLES = {
    "sinhala_mmlu": {
        "unicode": [
            {"question": "ලෝකයේ විශාලතම මහාද්වීපය කුමක්ද?",
             "options": ["ආසියාව", "අප්‍රිකාව", "යුරෝපය", "ඕස්ට්‍රේලියාව"],
             "answer_digit": "1"},
            {"question": "දිනකට පැය කීයක් තිබේද?",
             "options": ["12", "24", "48", "60"],
             "answer_digit": "2"},
        ],
        "romanized": [
            {"question": "lookayee wishaalathama mahaadwiipaya kumakda?",
             "options": ["aasiyaawa", "aprikaawa", "yuroopaya", "oostreeliyaawa"],
             "answer_digit": "1"},
            {"question": "dinakata paeya kiiyak thibeeda?",
             "options": ["12", "24", "48", "60"],
             "answer_digit": "2"},
        ],
    },
    "global_piqa": {
        "unicode": [
            {"question": "අත් සෝදා ගැනීමට වඩාත් සුදුසු දෙය කුමක්ද?",
             "options": ["සබන් හා වතුර භාවිතා කිරීම", "වැලි හා ගල් භාවිතා කිරීම"],
             "answer_digit": "1"},
            {"question": "රෑට කියවීමට වඩාත් සුදුසු ආලෝකය කුමක්ද?",
             "options": ["අඳුර", "පහන් එළිය"],
             "answer_digit": "2"},
        ],
        "romanized": [
            {"question": "ath soodaa gaeniimata wadaath sudusu deya kumakda?",
             "options": ["saban haa wathura bhaawithaa kiriima", "waeli haa gal bhaawithaa kiriima"],
             "answer_digit": "1"},
            {"question": "raaeta kiyawiimata wadaath sudusu aalookaya kumakda?",
             "options": ["andura", "pahan eliya"],
             "answer_digit": "2"},
        ],
    },
    "sold": {
        "unicode": [
            {"text": "අද කාලගුණය හොඳයි. එළියට ගිහින් ඇවිදින්න පුළුවන්.", "answer": "NOT"},
            {"text": "උඹ හරිම මෝඩයෙක්. කිසි දෙයක් තේරෙන්නෙ නෑ.", "answer": "OFF"},
        ],
        "romanized": [
            {"text": "ada kaalagunaya hondayi. eliyata gihin aewidinna puluwan.", "answer": "NOT"},
            {"text": "umba harima moodayek. kisi deyak theerennee naae.", "answer": "OFF"},
        ],
    },
}


def build_mcq_prompt_t2(question, options, dataset_key, script, subject=None):
    options_str, digit_labels = _mcq_options_block(options)
    n = len(options)
    valid_range = "/".join(digit_labels)
    examples = FEWSHOT_EXAMPLES[dataset_key][script]

    if subject is not None:
        instruction = (
            f"This is a multiple-choice question related to the {_humanize_domain(subject) if subject else ''}. "
            f"Choose the correct or most appropriate answer from answers {valid_range}. "
            f"{NO_REPEAT_INSTRUCTION} Respond with only the number ({valid_range}) and nothing else, "
            f"exactly like the examples below."
        )
    else:
        instruction = (
            f"Out of the given {n} answers, which one is the best option to complete the given text? "
            f"{NO_REPEAT_INSTRUCTION} Respond with only the number ({valid_range}) and nothing else, "
            f"exactly like the examples below."
        )

    shots = []
    for ex in examples:
        ex_options_str, _ = _mcq_options_block(ex["options"])
        shots.append(
            f"Question: {ex['question']}\n{ex_options_str}\nAnswer: {ex['answer_digit']}"
        )
    shots_block = "\n\n".join(shots)

    prompt = (
        f"{instruction}\n\n"
        f"{shots_block}\n\n"
        f"Question: {question}\n"
        f"{options_str}\n"
        f"Answer:"
    )
    return prompt, digit_labels


def build_sold_prompt_t2(text, script):
    examples = FEWSHOT_EXAMPLES["sold"][script]
    instruction = (
        "Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' "
        f"(offensive). {NO_REPEAT_INSTRUCTION} Respond with only 'NOT' or 'OFF' and nothing "
        "else, exactly like the examples below."
    )
    shots = [f"Text: {ex['text']}\nAnswer: {ex['answer']}" for ex in examples]
    shots_block = "\n\n".join(shots)
    return (
        f"{instruction}\n\n"
        f"{shots_block}\n\n"
        f"Text: {text}\n"
        f"Answer:"
    )


# ------------------------------------------------------------------
# T3 — ANSWER FIRST: state the answer before any reasoning
# ------------------------------------------------------------------

def build_mcq_prompt_t3(question, options, subject=None):
    options_str, digit_labels = _mcq_options_block(options)
    n = len(options)
    valid_range = "/".join(digit_labels)
    if subject is not None:
        prompt = (
            f"This is a multiple-choice question related to the {_humanize_domain(subject)}. "
            f"Choose the correct or most appropriate answer from answers {valid_range} for the "
            f"following question. {NO_REPEAT_INSTRUCTION} State the answer number "
            f"({valid_range}) FIRST, before any explanation. You may briefly explain your "
            f"reasoning only after giving the number.\n"
            f"Question: {question}\n"
            f"{options_str}\n"
            f"Answer:"
        )
    else:
        prompt = (
            f"Out of the given {n} answers, which one is the best option to complete the given "
            f"text? {NO_REPEAT_INSTRUCTION} State the answer number ({valid_range}) FIRST, "
            f"before any explanation. You may briefly explain your reasoning only after giving "
            f"the number.\n"
            f"{question}\n"
            f"{options_str}\n"
            f"Answer:"
        )
    return prompt, digit_labels


def build_sold_prompt_t3(text):
    return (
        f"Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' "
        f"(offensive). {NO_REPEAT_INSTRUCTION} State 'NOT' or 'OFF' FIRST, before any "
        f"explanation. You may briefly explain your reasoning only after giving the label.\n"
        f"Text: {text}\n"
        f"Answer:"
    )


# ------------------------------------------------------------------
# Dispatch — every call site goes through these, keyed off PROMPT_TEMPLATE
# ------------------------------------------------------------------

def build_mcq_prompt(template, question, options, dataset_key, script, subject=None):
    if template == "T1_direct":
        return build_mcq_prompt_t1(question, options, subject=subject)
    if template == "T2_fewshot":
        return build_mcq_prompt_t2(question, options, dataset_key, script, subject=subject)
    if template == "T3_answer_first":
        return build_mcq_prompt_t3(question, options, subject=subject)
    raise ValueError(f"Unknown template: {template}")


def build_sold_prompt(template, text, script):
    if template == "T1_direct":
        return build_sold_prompt_t1(text)
    if template == "T2_fewshot":
        return build_sold_prompt_t2(text, script)
    if template == "T3_answer_first":
        return build_sold_prompt_t3(text)
    raise ValueError(f"Unknown template: {template}")


# ============================================================
# LABEL EXTRACTION
# ============================================================

def _extract_token(generated_text, valid_tokens, keyword_pattern):
    """
    Shared boundary-safe extractor for both digit tokens ("1".."N") and word tokens
    ("NOT"/"OFF"). `keyword_pattern` is the regex alternation for words that typically
    precede the answer (e.g. "answer|option|choice" or "label|answer|classification").
    """
    if not generated_text:
        return "INVALID"
    text = generated_text.strip()
    pattern_tokens = "|".join(re.escape(t) for t in valid_tokens)
    is_digit = valid_tokens and valid_tokens[0].isdigit()
    boundary = r"(?<!\d)" if is_digit else r"\b"
    end_boundary = r"(?!\d)" if is_digit else r"\b"

    # 1. Leading answer: "1", "1.", "(1)", "**1**", "NOT", "Answer: NOT"
    prefix_match = re.match(
        rf"^(?:(?:the\s+)?(?:{keyword_pattern})\s*(?:is|=|:)?\s*)?"
        rf"[\*\(#\[\s]*{boundary}({pattern_tokens}){end_boundary}",
        text, re.IGNORECASE
    )
    if prefix_match:
        return prefix_match.group(1).upper()

    # 2. Explicit declaration in the body: "the answer is 2", "answer is NOT"
    body_match = re.search(
        rf"(?:(?:the\s+)?(?:{keyword_pattern})\s*(?:is|=|:)?\s*)"
        rf"[\*\(#\[\s]*{boundary}({pattern_tokens}){end_boundary}",
        text, re.IGNORECASE
    )
    if body_match:
        return body_match.group(1).upper()

    # 3. Standalone token anywhere, with a boundary on both sides.
    standalone_match = re.search(rf"{boundary}({pattern_tokens}){end_boundary}", text, re.IGNORECASE)
    if standalone_match:
        return standalone_match.group(1).upper()

    return "INVALID"


def extract_mcq_choice(generated_text, digit_labels):
    """Extract the chosen digit ("1".."N") from generated text."""
    return _extract_token(generated_text, digit_labels, r"correct\s+answer|answer|option|choice")


def digit_to_letter(digit):
    """"1" -> "A", "2" -> "B", ... "INVALID" passes through unchanged."""
    if digit.isdigit() and 1 <= int(digit) <= 26:
        return chr(64 + int(digit))
    return digit


def extract_sold_label(generated_text):
    """Extract NOT or OFF with whole-word boundary checks."""
    return _extract_token(generated_text, ["NOT", "OFF"], "label|answer|classification")


print(f"Prompt templates and extraction functions defined. Active PROMPT_TEMPLATE = {PROMPT_TEMPLATE!r}")

# Sanity check
p_mcq, digits = build_mcq_prompt(PROMPT_TEMPLATE, "Sample question?", ["opt1", "opt2", "opt3", "opt4"],
                                  "sinhala_mmlu", "unicode", subject="Social_Science")
print(f"\n--- Sample MMLU Prompt ---\n{p_mcq}")
p_piqa, _ = build_mcq_prompt(PROMPT_TEMPLATE, "Sample question?", ["opt1", "opt2"],
                              "global_piqa", "unicode")
print(f"\n--- Sample PIQA Prompt ---\n{p_piqa}")
print(f"\n--- Sample SOLD Prompt ---\n{build_sold_prompt(PROMPT_TEMPLATE, 'Sample text', 'unicode')}")
print(f"\ndigit_to_letter test: 1->{digit_to_letter('1')} 2->{digit_to_letter('2')} INVALID->{digit_to_letter('INVALID')}")

Prompt templates and extraction functions defined. Active PROMPT_TEMPLATE = 'T1_direct'

--- Sample MMLU Prompt ---
This is a multiple-choice question related to the Social Science. Choose the correct or most appropriate answer from answers 1/2/3/4 for the following question. Do not repeat the question or text in your response. Respond with only the number (1/2/3/4) and nothing else.
Question: Sample question?
1. opt1
2. opt2
3. opt3
4. opt4
Answer:

--- Sample PIQA Prompt ---
Out of the given 2 answers, which one is the best option to complete the given text? Do not repeat the question or text in your response. Respond with only the number (1/2) and nothing else.
Sample question?
1. opt1
2. opt2
Answer:

--- Sample SOLD Prompt ---
Classify the following Sinhala text as either 'NOT' (not offensive) or 'OFF' (offensive). Do not repeat the question or text in your response. Respond with only 'NOT' or 'OFF' and nothing else.
Text: Sample text
Answer:

digit_to_letter test: 1->A 2->B INVAL

## 6. Generation Engine with Native Chat Template
`REASONING_MODE = "none"` — see Configuration cell for details.
The engine applies this setting identically across both script conditions and all datasets.
A quick sanity-check generation is run immediately after definition.

In [7]:
@torch.inference_mode()
def generate_response(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """Generate a response from the instruct model given a plain-text prompt.

    REASONING_MODE (global, set in Configuration cell) controls how the chat
    template is applied for models that default to emitting <think>...</think> blocks:
      'none'                 — standard single user message (no suppression)
      'no_think'             — SmolLM3-style: adds '/no_think' as a system message
      'enable_thinking_false' — Qwen3.5-style: passes enable_thinking=False kwarg
    The setting is applied identically to both script conditions and all datasets.
    """
    has_template = hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None)

    try:
        if has_template:
            if REASONING_MODE == "no_think":
                messages = [
                    {"role": "system", "content": "/no_think"},
                    {"role": "user", "content": prompt},
                ]
                formatted_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            elif REASONING_MODE == "enable_thinking_false":
                messages = [{"role": "user", "content": prompt}]
                formatted_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
                )
            else:  # "none" — standard path
                messages = [{"role": "user", "content": prompt}]
                formatted_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
        else:
            formatted_prompt = prompt

        device = next(model.parameters()).device
        inputs = tokenizer(
            formatted_prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS
        ).to(device)
        input_len = inputs["input_ids"].shape[1]

        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": temperature > 0,
            "pad_token_id": tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        }
        if temperature > 0:
            gen_kwargs["temperature"] = temperature

        outputs = model.generate(**inputs, **gen_kwargs)
        generated_ids = outputs[0][input_len:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return generated_text, input_len, len(generated_ids)
    except Exception as e:
        torch.cuda.empty_cache()
        print(f"\n[WARNING] Generation error on prompt: {e}")
        return f"ERROR: {e}", 0, 0


print(f"Generation engine initialized. REASONING_MODE = {REASONING_MODE!r}")

# Quick sanity-check generation
_sample_item_text = "Sample question?"
_sample_prompt, _ = build_mcq_prompt(PROMPT_TEMPLATE, _sample_item_text, ["opt1", "opt2", "opt3", "opt4"],
                                      "sinhala_mmlu", "unicode", subject="Science")
_sample_out, _, _ = generate_response(_sample_prompt)
print(f"Sample output: {repr(_sample_out[:150])}")
if REASONING_MODE != "none" and "<think>" in _sample_out.lower():
    print("WARNING: <think> block still present — the suppression flag did not take effect "
          "for this model/transformers version. Investigate before trusting results.")

Generation engine initialized. REASONING_MODE = 'none'
Sample output: 'Question: What is the chemical formula for water?\n1. H2O\n2. CO2\n3. NH3\n4. CH4\nAnswer: 1. H'


## 7. Pilot Run

In [8]:
print("=" * 75)
print("PILOT RUN — 5 items per task per script condition")
print(f"Prompt template: {PROMPT_TEMPLATE}")
print(f"Reasoning mode:  {REASONING_MODE}")
print("=" * 75)

print("\n--- Sinhala MMLU (MCQ, numeric answer) ---")
for item in mmlu_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt, digits = build_mcq_prompt(PROMPT_TEMPLATE, s["text"], s["options"], "sinhala_mmlu", script,
                                           subject=item.get("strata", {}).get("domain"))
        gen_text, n_in, n_out = generate_response(prompt)
        pred_digit = extract_mcq_choice(gen_text, digits)
        pred_label = digit_to_letter(pred_digit)
        correct = "\u2713" if pred_label == item["label"] else "\u2717"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred_digit}->{pred_label} {correct} | raw={gen_text[:60]!r}")

print("\n--- SOLD (Binary) ---")
for item in sold_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt = build_sold_prompt(PROMPT_TEMPLATE, s["text"], script)
        gen_text, n_in, n_out = generate_response(prompt)
        pred = extract_sold_label(gen_text)
        correct = "\u2713" if pred == item["label"] else "\u2717"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred} {correct} | raw={gen_text[:60]!r}")

print("\n--- Global PIQA (MCQ, numeric answer) ---")
for item in piqa_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        prompt, digits = build_mcq_prompt(PROMPT_TEMPLATE, s["text"], s["options"], "global_piqa", script)
        gen_text, n_in, n_out = generate_response(prompt)
        pred_digit = extract_mcq_choice(gen_text, digits)
        pred_label = digit_to_letter(pred_digit)
        correct = "\u2713" if pred_label == item["label"] else "\u2717"
        print(f"  [{script:9}] id={item['id']} | gold={item['label']} pred={pred_digit}->{pred_label} {correct} | raw={gen_text[:60]!r}")

print("\n" + "=" * 75)
print("Pilot complete. Check output above for clean extractions before running Section 8.")
print("=" * 75)

PILOT RUN — 5 items per task per script condition
Prompt template: T1_direct
Reasoning mode:  none

--- Sinhala MMLU (MCQ, numeric answer) ---
  [unicode  ] id=mmlu_0001 | gold=A pred=4->D ✗ | raw='4. ආහාර පූජා හා වස්ත්\u200dර ��'
  [romanized] id=mmlu_0001 | gold=A pred=4->D ✗ | raw='4 is the correct answer.'
  [unicode  ] id=mmlu_0002 | gold=C pred=4->D ✗ | raw='4. හජ්ජි ඉතුෂවය (D)'
  [romanized] id=mmlu_0002 | gold=C pred=1->A ✗ | raw='1 is the correct answer. The correct answer is: naththal uth'
  [unicode  ] id=mmlu_0003 | gold=A pred=4->D ✗ | raw='4. สඳ බැස ගිය පසුව ය. (Answer is incorrect as the'
  [romanized] id=mmlu_0003 | gold=A pred=4->D ✗ | raw='4. sanda baesa giya pasuwa ya.'
  [unicode  ] id=mmlu_0004 | gold=B pred=4->D ✗ | raw='4. ශිව දෙවස්ටට ඉපතයි. (Note:'
  [romanized] id=mmlu_0004 | gold=B pred=4->D ✗ | raw='4. shiwa dew upathayi.'
  [unicode  ] id=mmlu_0005 | gold=C pred=1->A ✗ | raw='1 is the correct answer. The question is in Sinhala, and the'
  [romanized] id=mmlu

## 8. Full Evaluation

In [9]:
MCQ_FIELDNAMES_BASE = [
    "id", "gold_label",
    "unicode_pred_digit", "unicode_pred_label", "unicode_correct", "unicode_raw_output",
    "unicode_n_input_tokens", "unicode_n_output_tokens",
    "romanized_pred_digit", "romanized_pred_label", "romanized_correct", "romanized_raw_output",
    "romanized_n_input_tokens", "romanized_n_output_tokens",
]
SOLD_FIELDNAMES_BASE = [
    "id", "gold_label",
    "unicode_pred", "unicode_correct", "unicode_raw_output",
    "unicode_n_input_tokens", "unicode_n_output_tokens",
    "romanized_pred", "romanized_correct", "romanized_raw_output",
    "romanized_n_input_tokens", "romanized_n_output_tokens",
]

GLOBAL_START_TIME = time.time()

def _check_time_limit():
    """Return True if time limit in hours has been exceeded."""
    elapsed_hours = (time.time() - GLOBAL_START_TIME) / 3600.0
    return elapsed_hours >= TIME_LIMIT_HOURS


def _find_checkpoint_file(filename):
    """Find existing CSV in current working dir or in mounted /kaggle/input/ datasets."""
    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        return filename
    input_base = "/kaggle/input"
    if os.path.exists(input_base):
        for root, _, files in os.walk(input_base):
            if filename in files:
                candidate = os.path.join(root, filename)
                if os.path.getsize(candidate) > 0:
                    return candidate
    return None


def _load_existing_progress(output_csv, fieldnames):
    """
    If output_csv exists locally or in /kaggle/input, prepare output_csv with existing rows
    and return the set of already completed IDs.
    """
    if not RESUME:
        return set()
    
    src = _find_checkpoint_file(output_csv)
    if not src:
        return set()
    
    done_ids = set()
    rows = []
    try:
        with open(src, "r", newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for r in reader:
                if "id" in r and r["id"]:
                    done_ids.add(r["id"])
                    rows.append(r)
        
        if src != output_csv and rows:
            with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                for r in rows:
                    writer.writerow({k: r.get(k, "") for k in fieldnames})
                f.flush()
            print(f"[RESUME] Loaded {len(done_ids):,} existing records from {src} -> {output_csv}")
        elif src == output_csv:
            print(f"[RESUME] Found existing {output_csv} with {len(done_ids):,} records")
    except Exception as e:
        print(f"[WARNING] Could not read existing checkpoint from {src}: {e}")
        return set()
    
    return done_ids


def run_mcq_evaluation(dataset, dataset_name, output_csv, strata_fields, dataset_key, subject_field=None):
    """
    Run MCQ evaluation (MMLU or PIQA) on both script conditions.
    strata_fields: keys inside each item's 'strata' dict to copy into the CSV as columns.
    dataset_key: 'sinhala_mmlu' or 'global_piqa'.
    subject_field: key inside item['strata'] holding the MMLU subject/domain, or None for PIQA.
    """
    fieldnames = strata_fields + MCQ_FIELDNAMES_BASE
    done_ids = _load_existing_progress(output_csv, fieldnames)
    mode = "a" if (os.path.exists(output_csv) and os.path.getsize(output_csv) > 0) else "w"

    print(f"\n{'='*65}")
    print(f"Evaluating: {dataset_name} ({len(dataset):,} items, {len(done_ids):,} already done)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    remaining = [it for it in dataset if it["id"] not in done_ids]
    if not remaining:
        print(f"All {len(dataset):,} items already completed in {output_csv}.")
        return output_csv

    with open(output_csv, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if mode == "w":
            writer.writeheader()
            f.flush()

        t0 = time.time()
        timeout_hit = False
        n_processed = 0

        for i, item in enumerate(tqdm(remaining, desc=dataset_name)):
            if _check_time_limit():
                elapsed_h = (time.time() - GLOBAL_START_TIME) / 3600.0
                print(f"\n[TIME LIMIT GUARD] Elapsed: {elapsed_h:.2f}h >= {TIME_LIMIT_HOURS}h.")
                print(f"Saving checkpoint ({len(done_ids) + i:,}/{len(dataset):,} completed) and exiting cleanly.")
                timeout_hit = True
                break

            strata = item.get("strata", {})
            row = {k: strata.get(k) for k in strata_fields}
            row["id"] = item["id"]
            row["gold_label"] = item["label"]

            for script in ["unicode", "romanized"]:
                s = item[script]
                subject = strata.get(subject_field) if subject_field else None
                prompt, digits = build_mcq_prompt(PROMPT_TEMPLATE, s["text"], s["options"], dataset_key, script,
                                                   subject=subject)
                gen_text, n_in, n_out = generate_response(prompt)
                pred_digit = extract_mcq_choice(gen_text, digits)
                pred_label = digit_to_letter(pred_digit)
                correct = int(pred_label == item["label"])

                row[f"{script}_pred_digit"] = pred_digit
                row[f"{script}_pred_label"] = pred_label
                row[f"{script}_correct"] = correct
                row[f"{script}_raw_output"] = gen_text.strip()[:200]
                row[f"{script}_n_input_tokens"] = n_in
                row[f"{script}_n_output_tokens"] = n_out

            writer.writerow(row)
            f.flush()
            n_processed += 1

        elapsed = time.time() - t0

    if n_processed:
        print(f"Done {n_processed:,} new items in {elapsed/60:.1f} min ({elapsed/n_processed:.2f} s/item)")
    return output_csv


def run_sold_evaluation(dataset, output_csv):
    """Run SOLD binary classification on both script conditions."""
    fieldnames = ["label"] + SOLD_FIELDNAMES_BASE
    done_ids = _load_existing_progress(output_csv, fieldnames)
    mode = "a" if (os.path.exists(output_csv) and os.path.getsize(output_csv) > 0) else "w"

    print(f"\n{'='*65}")
    print(f"Evaluating: SOLD ({len(dataset):,} items, {len(done_ids):,} already done)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    remaining = [it for it in dataset if it["id"] not in done_ids]
    if not remaining:
        print(f"All {len(dataset):,} items already completed in {output_csv}.")
        return output_csv

    with open(output_csv, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if mode == "w":
            writer.writeheader()
            f.flush()

        t0 = time.time()
        timeout_hit = False
        n_processed = 0

        for i, item in enumerate(tqdm(remaining, desc="SOLD")):
            if _check_time_limit():
                elapsed_h = (time.time() - GLOBAL_START_TIME) / 3600.0
                print(f"\n[TIME LIMIT GUARD] Elapsed: {elapsed_h:.2f}h >= {TIME_LIMIT_HOURS}h.")
                print(f"Saving checkpoint ({len(done_ids) + i:,}/{len(dataset):,} completed) and exiting cleanly.")
                timeout_hit = True
                break

            row = {"id": item["id"], "gold_label": item["label"], "label": item["label"]}

            for script in ["unicode", "romanized"]:
                s = item[script]
                prompt = build_sold_prompt(PROMPT_TEMPLATE, s["text"], script)
                gen_text, n_in, n_out = generate_response(prompt)
                pred = extract_sold_label(gen_text)
                correct = int(pred == item["label"])

                row[f"{script}_pred"] = pred
                row[f"{script}_correct"] = correct
                row[f"{script}_raw_output"] = gen_text.strip()[:200]
                row[f"{script}_n_input_tokens"] = n_in
                row[f"{script}_n_output_tokens"] = n_out

            writer.writerow(row)
            f.flush()
            n_processed += 1

        elapsed = time.time() - t0

    if n_processed:
        print(f"Done {n_processed:,} new items in {elapsed/60:.1f} min ({elapsed/n_processed:.2f} s/item)")
    return output_csv


print("Evaluation runners ready with continuous flushing, checkpointing, and time guard.")

Evaluation runners ready with continuous flushing, checkpointing, and time guard.


In [10]:
# ============================================================
# RUN ALL THREE EVALUATIONS
# ============================================================

mmlu_csv = run_mcq_evaluation(
    mmlu_data, "Sinhala MMLU", f"{OUTPUT_PREFIX}_sinhala_mmlu.csv",
    strata_fields=["domain", "difficulty"], dataset_key="sinhala_mmlu", subject_field="domain"
)

sold_csv = run_sold_evaluation(
    sold_data, f"{OUTPUT_PREFIX}_sold.csv"
)

piqa_csv = run_mcq_evaluation(
    piqa_data, "Global PIQA", f"{OUTPUT_PREFIX}_global_piqa.csv",
    strata_fields=["culturally_specific"], dataset_key="global_piqa", subject_field=None
)

print("\n" + "="*65)
print("ALL EVALUATION LOOPS FINISHED")
print("="*65)


Evaluating: Sinhala MMLU (6,879 items, 0 already done)
Output:     zephyr_7b_beta_sinhala_mmlu.csv


Sinhala MMLU: 100%|██████████| 6879/6879 [8:46:55<00:00,  4.60s/it]


Done 6,879 new items in 526.9 min (4.60 s/item)

Evaluating: SOLD (2,500 items, 0 already done)
Output:     zephyr_7b_beta_sold.csv


SOLD: 100%|██████████| 2500/2500 [56:57<00:00,  1.37s/it]


Done 2,500 new items in 57.0 min (1.37 s/item)

Evaluating: Global PIQA (100 items, 0 already done)
Output:     zephyr_7b_beta_global_piqa.csv


Global PIQA: 100%|██████████| 100/100 [07:47<00:00,  4.67s/it]

Done 100 new items in 7.8 min (4.67 s/item)

ALL EVALUATION LOOPS FINISHED


## 9. Overall Summary

In [11]:
import pandas as pd

def summarize(csv_path, task_name):
    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        print(f"{task_name:14} -- Not evaluated or empty yet")
        return None
    try:
        df = pd.read_csv(csv_path)
        n = len(df)
        if n == 0:
            print(f"{task_name:14} n=0 (empty)")
            return None
        u_acc = df["unicode_correct"].mean() * 100
        r_acc = df["romanized_correct"].mean() * 100
        u_invalid_col = "unicode_pred_label" if "unicode_pred_label" in df.columns else "unicode_pred"
        r_invalid_col = "romanized_pred_label" if "romanized_pred_label" in df.columns else "romanized_pred"
        u_invalid = (df[u_invalid_col] == "INVALID").mean() * 100
        r_invalid = (df[r_invalid_col] == "INVALID").mean() * 100
        gap = u_acc - r_acc

        print(f"{task_name:14} n={n:5,}  Unicode={u_acc:6.2f}%  Romanized={r_acc:6.2f}%  "
              f"Gap={gap:+6.2f}pp  Invalid(U/R)={u_invalid:.1f}%/{r_invalid:.1f}%")
        return {"Task": task_name, "N": n, "Unicode Acc (%)": round(u_acc, 2),
                "Romanized Acc (%)": round(r_acc, 2), "Gap (pp)": round(gap, 2),
                "Unicode Invalid (%)": round(u_invalid, 2), "Romanized Invalid (%)": round(r_invalid, 2)}
    except Exception as e:
        print(f"[WARNING] Could not summarize {csv_path}: {e}")
        return None


print(f"{MODEL_LABEL} \u2014 Extrinsic Evaluation Summary")
print("-" * 90)
raw_results = [
    summarize(mmlu_csv, "Sinhala MMLU"),
    summarize(sold_csv, "SOLD"),
    summarize(piqa_csv, "Global PIQA"),
]
results = [r for r in raw_results if r is not None]
if results:
    summary_df = pd.DataFrame(results)
    summary_csv = f"{OUTPUT_PREFIX}_extrinsic_summary.csv"
    summary_df.to_csv(summary_csv, index=False)
    print(f"\nSummary saved to: {summary_csv}")
    display(summary_df) if "display" in globals() else print(summary_df)
else:
    print("\nNo completed evaluation results to summarize yet.")

zephyr-7b-beta — Extrinsic Evaluation Summary
------------------------------------------------------------------------------------------
Sinhala MMLU   n=6,879  Unicode= 24.31%  Romanized= 24.89%  Gap= -0.58pp  Invalid(U/R)=2.1%/0.6%
SOLD           n=2,500  Unicode= 59.72%  Romanized= 59.76%  Gap= -0.04pp  Invalid(U/R)=0.2%/0.1%
Global PIQA    n=  100  Unicode= 48.00%  Romanized= 50.00%  Gap= -2.00pp  Invalid(U/R)=0.0%/0.0%

Summary saved to: zephyr_7b_beta_extrinsic_summary.csv
           Task     N  Unicode Acc (%)  Romanized Acc (%)  Gap (pp)  \
0  Sinhala MMLU  6879            24.31              24.89     -0.58   
1          SOLD  2500            59.72              59.76     -0.04   
2   Global PIQA   100            48.00              50.00     -2.00   

   Unicode Invalid (%)  Romanized Invalid (%)  
0                 2.11                   0.64  
1                 0.20                   0.12  
2                 0.00                   0.00  


## 10. Output File Listing

In [12]:
print("All output files in working directory:")
for f in [mmlu_csv, sold_csv, piqa_csv]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f"  {f} ({size:.1f} KB)")
    else:
        print(f"  {f} (not created yet)")

summary_file = f"{OUTPUT_PREFIX}_extrinsic_summary.csv"
if os.path.exists(summary_file):
    size = os.path.getsize(summary_file) / 1024
    print(f"  {summary_file} ({size:.1f} KB)")

print(f"\n\u2705 {MODEL_LABEL} extrinsic evaluation run completed!")

All output files in working directory:
  zephyr_7b_beta_sinhala_mmlu.csv (1292.1 KB)
  zephyr_7b_beta_sold.csv (188.0 KB)
  zephyr_7b_beta_global_piqa.csv (19.4 KB)
  zephyr_7b_beta_extrinsic_summary.csv (0.2 KB)

✅ zephyr-7b-beta extrinsic evaluation run completed!
